# Issue #358 — European-average-odds value proxy

Executed from the frozen #354 10h pre-outcome feature surface and a hash-bound European 1X2 source. The `10h` field is a retrospective relative horizon, not an exact observation timestamp or Beijing SP.

In [1]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

package = Path('/home/drwho1985/worktrees/bj358/results/low_n_eu_value_proxy')
cells = pq.read_table(package / 'value_cells.parquet').to_pandas()
quality = pd.read_csv(package / 'quality_by_fixture.csv')
profile = pd.read_csv(package / 'year_profile.csv')
recipe = pd.read_csv(package / 'n3_n5_full_cartesian_recipe.csv')
receipt = json.loads((package / 'FREEZE_RECEIPT.json').read_text(encoding='utf-8'))
coverage = {
    'cells': int(len(cells)),
    'fixtures': int(cells[['issue', 'match_idx']].drop_duplicates().shape[0]),
    'issues': int(cells['issue'].nunique()),
    'years': sorted(int(value) for value in cells['issue_year'].unique()),
    'outcome_order': list(cells.groupby(['issue', 'match_idx'])['outcome_code'].agg(tuple).iloc[0]),
}
print(json.dumps(coverage, indent=2, sort_keys=True))

{
  "cells": 404466,
  "fixtures": 134822,
  "issues": 442,
  "outcome_order": [
    "3",
    "1",
    "0"
  ],
  "years": [
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ]
}


In [2]:
quality_summary = (
    quality.groupby('quality_grade', sort=True)
    .agg(fixtures=('match_idx', 'size'), mean_books=('contributing_bookmaker_count', 'mean'), invalid_rows=('invalid_bookmaker_rows', 'sum'))
    .reset_index()
)
print(quality_summary.to_string(index=False))

quality_grade  fixtures  mean_books  invalid_rows
            A     30733   15.000000             0
            B    103661   12.518401        257245
            C       428    3.128505          5081


In [3]:
print(profile.to_string(index=False))

 issue_year quality_grade  issues  fixtures  cells  mean_contributing_books  mean_missing_bookmaker_fraction  mean_overround  mean_top2_reliability
       2017             A      47       539  17232                15.000000                         0.000000        1.080435               0.740114
       2017             B      48       568  33294                12.215715                         0.185619        1.083653               0.740753
       2017             C      12        17     51                 2.529412                         0.831373        1.094926               0.765741
       2018             A      58       559  18309                15.000000                         0.000000        1.077406               0.748752
       2018             B      59       571  39309                12.248111                         0.183459        1.082204               0.748752
       2018             C      12        15     48                 2.312500                         0.845833    

In [4]:
print(recipe.to_string(index=False))

 n  outcomes_per_fixture  full_cartesian_ticket_count  ticket_level_payout_factor payout_factor_application                           gross_multiplier_formula  fixture_selection_performed                                             interpretation
 3                     2                            8                        0.65  ONCE_PER_COMPLETE_TICKET 0.65 * product(top2_full_cartesian_contribution_i)                        False ingredient only; no fixture subset or strategy is selected
 4                     2                           16                        0.65  ONCE_PER_COMPLETE_TICKET 0.65 * product(top2_full_cartesian_contribution_i)                        False ingredient only; no fixture subset or strategy is selected
 5                     2                           32                        0.65  ONCE_PER_COMPLETE_TICKET 0.65 * product(top2_full_cartesian_contribution_i)                        False ingredient only; no fixture subset or strategy is selected


In [5]:
forbidden = {'result_code', 'winning_sp', 'settlement_sp', 'payout', 'realized_gross'}
identities = {
    'average_odds_overround_all_positive': bool(cells['average_odds_overround'].gt(0).all()),
    'source_consensus_matches_354_raw': bool(cells['source_consensus_devig_probability'].gt(0).all()),
    'calibrated_probability_positive': bool(cells['calibrated_probability_354'].gt(0).all()),
    'no_settlement_columns': not bool(forbidden & set(cells.columns)),
    'relative_horizon_only': sorted(cells['source_timestamp_semantics'].unique().tolist()),
}
assert all(value is True for key, value in identities.items() if key != 'relative_horizon_only')
print(json.dumps(identities, indent=2, sort_keys=True))

{
  "average_odds_overround_all_positive": true,
  "calibrated_probability_positive": true,
  "no_settlement_columns": true,
  "relative_horizon_only": [
    "RELATIVE_10H_LABEL_NO_EXACT_OBSERVATION_TIMESTAMP"
  ],
  "source_consensus_matches_354_raw": true
}


In [6]:
print(json.dumps(receipt, indent=2, sort_keys=True))

{
  "contract": "BJ-UNIFIED-LOWN-EU-VALUE-DATA-001/v1",
  "fixture_selection_performed": false,
  "horizon": "10h",
  "n_recipe_sha256": "097c2d30f7136be72e68adb11d7ba21e0adb173775c58e558a056409f8242fd2",
  "outcome_order": [
    "3",
    "1",
    "0"
  ],
  "payout_factor": 0.65,
  "quality_sha256": "49cccb3a36db793d6ed3343dfe37d9e003db826eee8b0ff6d2419179457b3fc5",
  "source_timestamp_semantics": "RELATIVE_10H_LABEL_NO_EXACT_OBSERVATION_TIMESTAMP",
  "value_cell_count": 404466,
  "value_cells_sha256": "6adc99a6c001f01b155314cc0dbca0694eddf055187dea6cbfd22200c50b6bb1",
  "year_profile_sha256": "4b22d0700934ffbdddcaf94e875659adc66b052fdc2f2a8fe77d52f0f10191e1"
}


## Decision boundary

The package exposes odds-derived value ingredients for later low-N research. It does not use settlement labels, calibrate Beijing payout, rank fixtures, choose a selector, or claim an executable betting edge.